# Swarm data access
This tutorial demonstrates how to access Swarm data using GeospaceLAB. We will use the Swarm MAG LR dataset as an example, but the same process applies to other datasets as well.
A full list of the supported datasets and their name patterns can be found in the [Supported datasets](2_supported_datasets.ipynb) tutorial. Also, the examples for accessing other datasets can be found in the [Examples](example_Swarm_MAG.ipynb) tutorial.

:::{tip}
In addtion to the examples shown in this documentation, more example scripts can be found in GeospaceLAB [examples](https://github.com/JouleCai/geospacelab/tree/master/examples) and [test](https://github.com/JouleCai/geospacelab/tree/master/test).
:::

The Swarm data can be accessed in three ways:
1. [via Swarm Dissemination Server](#swarm-dissemination-server)
2. [via Swarm VirES](#swarm-vires)
3. [via Swarm VirES HAPI Service](#swarm-hapi)

:::{note}
For 2 and 3, GeospaceLAB currently supports only the MAG LR data. Supports for other data products will be implemented next.
:::

In [1]:
import datetime
import geospacelab.datahub as datahub

(swarm-dissemination-server)=
## Access data via Swarm Dissemination Sever

### Initial settings

In [2]:
dt_fr = datetime.datetime(2016, 3, 15, 18, 18)
dt_to = datetime.datetime(2016, 3, 15, 18, 23)
sat_id = "A"

### Create a datahub

In [3]:
dh = datahub.DataHub(
    dt_fr=dt_fr,    # Start time for data retrieval
    dt_to=dt_to,    # End time for data retrieval
    visual='on'     # Enable plotting properties for datasets and variables. 
                    # Used for visualizing the data in the dashboard.
    )

:::{tip}
Above returns a `DataHub` object, which is used as a data manager in GeospaceLAB. A `DataHub` object (here is `dh`) can manage multiple datasets (GeospaceLAB `Dataset` objects) and a `Dataset` object manages multiple variables (GeospaceLAB `Variable` objects). 
:::

### Dock a Swarm dataset

The `dock` method is called to dock a specific dataset. The `dock` method responds to a sequence of work flows, including
- contacting the Swarm Dissemination FTP Sever,
- checking the data product version and data availability,
- downloading the data files,
- loading the data files to the dataset.

The method takes the name patterns of the dataset as an argument and returns a dataset object. For example, the following code docks the Swarm MAG LR dataset for the specified time range and satellite ID as shwon above. A full list of the supported datasets and required inputs can be found in the [Supported datasets](2_supported_datasets.ipynb) tutorial.

In [4]:
ds = dh.dock(
    datasource_contents=['esa_eo', 'swarm', 'l1b', 'mag_lr'],  # The name patterns of the data products to be retrieved.
    sat_id=sat_id,  # The Swarm satellite ID (A, B, or C) for which the data products will be retrieved.
    product_version='latest',  # The version of the data products to be retrieved.
                            # Default is 'latest', which retrieves the most recent version available.
                            # To specify a particular version, use the format 'XXXX' (e.g., '0301').
    add_APEX=True,  # Whether to add APEX coordinates to the dataset.
    )

Searching the data product "MAG_LR" with the version "latest" on the server...
The file [PosixPath('/data/afys-ionosphere/data/ESA/SWARM/Level1b/MAG_LR/0701/Sat_A/2016/SW_OPER_MAGA_LR_1B_20160315T000000_20160315T235959_0701_MDR_MAG_LR.cdf'), PosixPath('/data/afys-ionosphere/data/ESA/SWARM/Level1b/MAG_LR/0701/Sat_A/2016/SW_OPER_MAGA_LR_1B_20160315T000000_20160315T235959_0701_ASM_VFM_IC.cdf')] already exists: skip downloading.
/opt/anaconda3/envs/Swarm/lib/python3.12/site-packages/numpy/_core/numeric.py:442: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


:::{note}
The data files downloaded from the Swarm Dissemination FTP Sever are stored in the GeospaceLAB data folder (see [](./1_configuration.md)). The data product with different versions are arranged in separate folders.
:::

:::{tip}
To access other supported Swarm data products, check [](./2_supported_datasets.ipynb) and replace the keyword `datasource_contents=['esa_eo', 'swarm', 'l1b', 'mag_lr']` with the corresponding name patterns. For example, to dock the Swarm EFI TCT 2Hz data, use the keyword `datasource_contents=['esa_eo', 'swarm', 'advanced', 'efi_tct02']`. This is also applied for other datasets supported in GeospaceLAB.
:::

The data files can be listed by calling

In [5]:
ds.data_file_paths

[PosixPath('/data/afys-ionosphere/data/ESA/SWARM/Level1b/MAG_LR/0701/Sat_A/2016/SW_OPER_MAGA_LR_1B_20160315T000000_20160315T235959_0701_MDR_MAG_LR.cdf')]

And the data file versions can be listed by calling

In [6]:
ds.data_file_versions

['0701']

#### List the variables included in the dataset

To see the variables loaded in the dataset, call 

In [7]:
ds.list_all_variables()

Dataset: esa/earthonline | swarm | mag | mag_lr
Printing all of the variables ...
|No.                 |Variable name                 |Variable name (Source)        |Description                                                                                         |
|--------------------|------------------------------|------------------------------|----------------------------------------------------------------------------------------------------|
|1                   |SC_DATETIME                   |N/A                           |Time of observation                                                                                 |
|2                   |SYNC_STATUS                   |SyncStatus                    |                                                                                                    |
|3                   |SC_GEO_LAT                    |Latitude                      |Geographic Latitude                                                                       

#### Get a variable and data array

In [8]:
B_N = ds['B_N']

Above returns a GeospaceLAB Variable object. To get the data array, call

In [9]:

B_N_arr = B_N.value

Above returns a `numpy.ndarray` data array. 

It is also posible to get the variable using the variable name in the source file/data as shown in the list above:

In [10]:
ds['SC_GEO_LAT'] == ds['Latitude']

True

(swarm-vires)=
## Access data via Swarm VirES (developing)

In a similar way, users can dock the Swarm data product from the Swarm VirES service, by simply adding the keyword `from_VirES=True`. It is also possible to access the Swarm FAST data by add the keyword `from_FAST=True`.

:::{note}
Support for the FAST data is only availble when `from_VirES=True` or `from_HAPI=True` is called.
:::

### Retreive data from VirES

In [11]:
ds_VirES = dh.dock(
    datasource_contents=['esa_eo', 'swarm', 'l1b', 'mag_lr'], 
    sat_id=sat_id,
    from_VirES=True,  # Whether to retrieve the data from VirES instead of the local cache. 
    )
ds_VirES.list_all_variables()

INFO: Loading data from VirES for collection SW_OPER_MAGA_LR_1B with products {'measurements': ['B_VFM', 'B_NEC', 'dB_Sun', 'dB_AOCS', 'dB_other', 'B_error', 'q_NEC_CRF', 'Att_error', 'Flags_B', 'Flags_q', 'Flags_Platform', 'Flags_F', 'ASM_Freq_Dev', 'F', 'F_error', 'dF_Sun', 'dF_AOCS', 'dF_other'], 'models': ['CHAOS-Core'], 'residuals': False}.
Processing:  100%|██████████|  [ Elapsed: 00:00, Remaining: 00:00 ] [1/1] 
Downloading: 100%|██████████|  [ Elapsed: 00:00, Remaining: 00:00 ] (0.219MB)
INFO: Data loaded from VirES.
/opt/anaconda3/envs/Swarm/lib/python3.12/site-packages/numpy/_core/numeric.py:442: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')
Dataset: esa/earthonline | swarm | mag | mag_lr
Printing all of the variables ...
|No.                 |Variable name                 |Variable name (Source)        |Description                                                                                         |
|------------

### Retrieve FAST data from VirES

In [16]:
dt_now = datetime.datetime.now()
dt_fr_FAST = datetime.datetime(dt_now.year, dt_now.month, dt_now.day) - datetime.timedelta(days=30)
dt_to_FAST = datetime.datetime(dt_now.year, dt_now.month, dt_now.day) - datetime.timedelta(days=30) \
    + datetime.timedelta(hours=0, minutes=2)
ds_VirES_FAST = dh.dock(
    datasource_contents=['esa_eo', 'swarm', 'l1b', 'mag_lr'], 
    dt_fr=dt_fr_FAST,    # Start time for data retrieval
    dt_to=dt_to_FAST,    # End time for data retrieval
    sat_id=sat_id,
    from_VirES=True,  # Whether to retrieve the data from VirES instead of the local cache. 
    from_FAST=True  # Whether to retrieve the data from FAST instead of the local cache.
    )
ds_VirES_FAST.list_all_variables()

INFO: Loading data from VirES for collection SW_FAST_MAGA_LR_1B with products {'measurements': ['B_VFM', 'B_NEC', 'dB_Sun', 'dB_AOCS', 'dB_other', 'B_error', 'q_NEC_CRF', 'Att_error', 'Flags_B', 'Flags_q', 'Flags_Platform', 'Flags_F', 'ASM_Freq_Dev', 'F', 'F_error', 'dF_Sun', 'dF_AOCS', 'dF_other'], 'models': ['CHAOS-Core'], 'residuals': False}.
Processing:  100%|██████████|  [ Elapsed: 00:00, Remaining: 00:00 ] [1/1] 
Downloading: 100%|██████████|  [ Elapsed: 00:00, Remaining: 00:00 ] (0.217MB)
INFO: Data loaded from VirES.
/opt/anaconda3/envs/Swarm/lib/python3.12/site-packages/numpy/_core/numeric.py:442: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')
Dataset: esa/earthonline | swarm | mag | mag_lr
Printing all of the variables ...
|No.                 |Variable name                 |Variable name (Source)        |Description                                                                                         |
|------------

(swarm-hapi)=
## Access data via Swarm VirES HAPI Service (developing)

Users can dock the Swarm data product from the VirES HAPI service, by simply adding the keyword `from_HAPI=True`. 

### Retrieve data from HAPI

In [ ]:
ds_HAPI = dh.dock(
    datasource_contents=['esa_eo', 'swarm', 'l1b', 'mag_lr'], 
    sat_id=sat_id,
    from_HAPI=True,  # Whether to retrieve the data from HAPI instead of the local cache. 
    )
ds_HAPI.list_all_variables()

INFO: Loading data from HAPI for server https://vires.services/hapi, dataset SW_OPER_MAGA_LR_1B, parameters Timestamp,Latitude,Longitude,Radius.
INFO: Data loaded from HAPI.
INFO: Loading data from HAPI for server https://vires.services/hapi, dataset SW_OPER_MAGA_LR_1B, parameters F,dF_Sun,dF_AOCS,dF_other,F_error,B_VFM,B_NEC,dB_Sun,dB_AOCS,dB_other,B_error.
INFO: Data loaded from HAPI.
INFO: Loading data from HAPI for server https://vires.services/hapi, dataset SW_OPER_MAGA_LR_1B, parameters q_NEC_CRF,Att_error,Flags_F,Flags_B,Flags_q,Flags_Platform,ASM_Freq_Dev.
INFO: Data loaded from HAPI.
INFO: Loading data from HAPI for server https://vires.services/hapi, dataset SW_OPER_MAGA_LR_1B, parameters SyncStatus,B_NEC_Model,F_Model,F_res_Model,B_NEC_res_Model.
INFO: Data loaded from HAPI.
/opt/anaconda3/envs/Swarm/lib/python3.12/site-packages/numpy/_core/numeric.py:476: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')
Dataset: esa/ea

### Retrieve FAST data from HAPI

In [17]:
dt_now = datetime.datetime.now()
dt_fr_FAST = datetime.datetime(dt_now.year, dt_now.month, dt_now.day) - datetime.timedelta(days=30)
dt_to_FAST = datetime.datetime(dt_now.year, dt_now.month, dt_now.day) - datetime.timedelta(days=30) \
    + datetime.timedelta(hours=0, minutes=2)
ds_HAPI_FAST = dh.dock(
    datasource_contents=['esa_eo', 'swarm', 'l1b', 'mag_lr'], 
    dt_fr=dt_fr_FAST,    # Start time for data retrieval
    dt_to=dt_to_FAST,    # End time for data retrieval
    sat_id=sat_id,
    from_HAPI=True,  # Whether to retrieve the data from HAPI instead of the local cache. 
    from_FAST=True  # Whether to retrieve the data from FAST instead of the local cache.
    )
ds_HAPI_FAST.list_all_variables()

INFO: Loading data from HAPI for server https://vires.services/hapi, dataset SW_FAST_MAGA_LR_1B, parameters Timestamp,Latitude,Longitude,Radius.
INFO: Data loaded from HAPI.
INFO: Loading data from HAPI for server https://vires.services/hapi, dataset SW_FAST_MAGA_LR_1B, parameters F,dF_Sun,dF_AOCS,dF_other,F_error,B_VFM,B_NEC,dB_Sun,dB_AOCS,dB_other,B_error.
INFO: Data loaded from HAPI.
INFO: Loading data from HAPI for server https://vires.services/hapi, dataset SW_FAST_MAGA_LR_1B, parameters q_NEC_CRF,Att_error,Flags_F,Flags_B,Flags_q,Flags_Platform,ASM_Freq_Dev.
INFO: Data loaded from HAPI.
INFO: Loading data from HAPI for server https://vires.services/hapi, dataset SW_FAST_MAGA_LR_1B, parameters SyncStatus,B_NEC_Model,F_Model,F_res_Model,B_NEC_res_Model.
INFO: Data loaded from HAPI.
/opt/anaconda3/envs/Swarm/lib/python3.12/site-packages/numpy/_core/numeric.py:442: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')
Dataset: esa/ea